# 01 - Data Audit and Quality Control

## پیش‌بینی قصد تداوم استفاده دانشجویان از یادگیری الکترونیکی با یادگیری ماشین

هدف این نوت‌بوک، ممیزی کیفیت داده‌ها، بررسی پرسشنامه‌ها و آماده‌سازی خروجی‌های لازم برای مراحل بعدی پروژه است.

### Dataset B
- نقش: Development / Training Dataset
- مسیر: `data/dataset_B`
- داده خام بدون تغییر نگهداری می‌شود و شاخص‌های کیفیت به‌صورت Flag ثبت می‌شوند.

### Dataset C
- مسیر: `data/excluded_datasets`
- به دلیل تکرار گسترده رکوردهای کامل، از تحلیل اصلی کنار گذاشته شده است؛ اما برای مستندسازی و reproducibility در پروژه نگهداری می‌شود.

### Dataset EFL
- نقش: External Validation Candidate
- مسیر: `data/datasets_EFL`
- داده خام بدون تغییر نگهداری می‌شود و پاسخ‌های مشکوک به‌صورت Quality Flag ثبت می‌شوند.

### اصل کنترل کیفیت
در تحلیل اصلی، Straight-line بودن به‌تنهایی به معنی حذف خودکار رکورد نیست. نسخه‌های فیلترشده فقط برای بررسی حساسیت/robustness نگهداری می‌شوند.


In [1]:
# در این سلول مسیر اصلی پروژه به‌صورت خودکار شناسایی و مسیر پوشه‌های داده و خروجی تعریف می‌شود.

from pathlib import Path
import pandas as pd
import numpy as np


# مسیر فعلی اجرای Notebook
CURRENT_DIR = Path.cwd().resolve()


# اگر Notebook از داخل پوشه notebooks اجرا شود، پوشه والد همان ریشه پروژه است.
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent

# اگر Jupyter از خود پوشه اصلی پروژه اجرا شده باشد، همان مسیر ریشه پروژه است.
elif (
    (CURRENT_DIR / "notebooks").exists()
    and (CURRENT_DIR / "data").exists()
):
    PROJECT_ROOT = CURRENT_DIR

else:
    raise FileNotFoundError(
        "Project root could not be detected. "
        "Please run this notebook from the project folder or the notebooks folder."
    )


DATASET_B_FOLDER = PROJECT_ROOT / "data" / "dataset_B"

DATASET_C_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "excluded_datasets"
)

DATASET_EFL_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "datasets_EFL"
)

PROCESSED_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

TABLES_FOLDER = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nDataset B folder:")
print(DATASET_B_FOLDER)

print("\nDataset C folder:")
print(DATASET_C_FOLDER)

print("\nDataset EFL folder:")
print(DATASET_EFL_FOLDER)

print("\nProcessed folder:")
print(PROCESSED_FOLDER)


# بررسی وجود مسیرهای اصلی پروژه
for folder_name, folder_path in {

    "Dataset B": DATASET_B_FOLDER,
    "Dataset C": DATASET_C_FOLDER,
    "Dataset EFL": DATASET_EFL_FOLDER,
    "Processed": PROCESSED_FOLDER

}.items():

    print(
        f"{folder_name}: "
        f"{folder_path} -> "
        f"Exists: {folder_path.exists()}"
    )

Project root:
F:\E_Learning_Continuance_ML

Dataset B folder:
F:\E_Learning_Continuance_ML\data\dataset_B

Dataset C folder:
F:\E_Learning_Continuance_ML\data\excluded_datasets

Dataset EFL folder:
F:\E_Learning_Continuance_ML\data\datasets_EFL

Processed folder:
F:\E_Learning_Continuance_ML\data\processed
Dataset B: F:\E_Learning_Continuance_ML\data\dataset_B -> Exists: True
Dataset C: F:\E_Learning_Continuance_ML\data\excluded_datasets -> Exists: True
Dataset EFL: F:\E_Learning_Continuance_ML\data\datasets_EFL -> Exists: True
Processed: F:\E_Learning_Continuance_ML\data\processed -> Exists: True


In [3]:
# در این سلول فایل خام Dataset B از مسیر مرکزی پروژه خوانده می‌شود.
# در این سلول کتابخانه‌های مورد نیاز و مسیرهای اصلی پروژه تعریف می‌شوند.

from pathlib import Path
import os
import pandas as pd
import numpy as np
file_B = os.path.join(DATASET_B_FOLDER, "pone.0308630.s002.csv")
df_B = pd.read_csv(file_B)
df_B.head()


,index,totalseconds,Gender,Age,Grade,Subject,PEdS1,PEdS2,PEdS3,PEdS4,...,PU3,CON1,CON2,CON3,SAT1,SAT2,SAT3,CI1,CI2,CI3
0,1,65,2,3,4,1,4,5,5,5,...,2,4,5,4,5,5,4,5,5,5
1,2,103,2,3,4,1,4,4,4,4,...,4,4,4,5,5,4,4,3,3,3
2,3,110,1,3,4,1,4,4,4,3,...,2,3,4,3,2,2,4,2,2,4
3,4,98,2,3,4,1,2,4,4,4,...,5,4,5,5,5,5,5,5,5,4
4,5,116,2,3,4,1,3,4,3,2,...,3,2,2,2,2,3,4,4,3,3


In [4]:
# در این سلول ابعاد Dataset B و نام تمام ستون‌های آن بررسی می‌شوند.

print("Shape of Dataset B:", df_B.shape)

print("\nColumns of Dataset B:")
for column in df_B.columns:
    print(column)

Shape of Dataset B: (368, 26)

Columns of Dataset B:
index
totalseconds
Gender
Age
Grade
Subject
PEdS1
PEdS2
PEdS3
PEdS4
PEmS1
PEmS2
PEmS3
PEmS4
PU1
PU2
PU3
CON1
CON2
CON3
SAT1
SAT2
SAT3
CI1
CI2
CI3


In [5]:
# در این سلول ستون‌های Dataset B بر اساس نقش و سازه پرسشنامه دسته‌بندی می‌شوند.

general_columns_B = [
    "index",
    "totalseconds",
    "Gender",
    "Age",
    "Grade",
    "Subject"
]

PEdS_columns_B = [
    "PEdS1",
    "PEdS2",
    "PEdS3",
    "PEdS4"
]

PEmS_columns_B = [
    "PEmS1",
    "PEmS2",
    "PEmS3",
    "PEmS4"
]

PU_columns_B = [
    "PU1",
    "PU2",
    "PU3"
]

CON_columns_B = [
    "CON1",
    "CON2",
    "CON3"
]

SAT_columns_B = [
    "SAT1",
    "SAT2",
    "SAT3"
]

CI_columns_B = [
    "CI1",
    "CI2",
    "CI3"
]

print("General columns:", general_columns_B)
print("PEdS columns:", PEdS_columns_B)
print("PEmS columns:", PEmS_columns_B)
print("PU columns:", PU_columns_B)
print("CON columns:", CON_columns_B)
print("SAT columns:", SAT_columns_B)
print("CI columns:", CI_columns_B)

General columns: ['index', 'totalseconds', 'Gender', 'Age', 'Grade', 'Subject']
PEdS columns: ['PEdS1', 'PEdS2', 'PEdS3', 'PEdS4']
PEmS columns: ['PEmS1', 'PEmS2', 'PEmS3', 'PEmS4']
PU columns: ['PU1', 'PU2', 'PU3']
CON columns: ['CON1', 'CON2', 'CON3']
SAT columns: ['SAT1', 'SAT2', 'SAT3']
CI columns: ['CI1', 'CI2', 'CI3']


In [6]:
# در این سلول نوع داده هر ستون در Dataset B بررسی می‌شود.

df_B.dtypes

index           int64
totalseconds    int64
Gender          int64
Age             int64
Grade           int64
Subject         int64
PEdS1           int64
PEdS2           int64
PEdS3           int64
PEdS4           int64
PEmS1           int64
PEmS2           int64
PEmS3           int64
PEmS4           int64
PU1             int64
PU2             int64
PU3             int64
CON1            int64
CON2            int64
CON3            int64
SAT1            int64
SAT2            int64
SAT3            int64
CI1             int64
CI2             int64
CI3             int64
dtype: object

In [7]:
# در این سلول تعداد مقادیر گمشده در هر ستون Dataset B بررسی می‌شود.

missing_B = df_B.isnull().sum()

print(missing_B)

index           0
totalseconds    0
Gender          0
Age             0
Grade           0
Subject         0
PEdS1           0
PEdS2           0
PEdS3           0
PEdS4           0
PEmS1           0
PEmS2           0
PEmS3           0
PEmS4           0
PU1             0
PU2             0
PU3             0
CON1            0
CON2            0
CON3            0
SAT1            0
SAT2            0
SAT3            0
CI1             0
CI2             0
CI3             0
dtype: int64


In [8]:
# در این سلول بررسی می‌شود که آیا رکورد کاملاً تکراری یا شناسه تکراری در Dataset B وجود دارد یا خیر.

duplicate_rows_B = df_B.duplicated().sum()
duplicate_index_B = df_B["index"].duplicated().sum()

print("Number of completely duplicated rows:", duplicate_rows_B)
print("Number of duplicated index values:", duplicate_index_B)

Number of completely duplicated rows: 0
Number of duplicated index values: 0


In [9]:
# در این سلول بررسی می‌شود که پاسخ‌های تمام آیتم‌های پرسشنامه فقط در بازه 1 تا 5 قرار داشته باشند.

questionnaire_columns_B = (
    PEdS_columns_B
    + PEmS_columns_B
    + PU_columns_B
    + CON_columns_B
    + SAT_columns_B
    + CI_columns_B
)

for column in questionnaire_columns_B:
    values = sorted(df_B[column].unique())
    print(column, ":", values)

PEdS1 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEdS2 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEdS3 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEdS4 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEmS1 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEmS2 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEmS3 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PEmS4 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PU1 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PU2 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
PU3 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
CON1 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
CON2 : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
CON3 : [np.int64(1), np.int64(2), np.int64(3),

In [10]:
# در این سلول تعداد پاسخ‌های 1 تا 5 برای هر سؤال پرسشنامه Dataset B محاسبه می‌شود.

response_counts_B = pd.DataFrame()

for column in questionnaire_columns_B:
    counts = df_B[column].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)

    response_counts_B.loc[column, "Count_1"] = counts[1]
    response_counts_B.loc[column, "Count_2"] = counts[2]
    response_counts_B.loc[column, "Count_3"] = counts[3]
    response_counts_B.loc[column, "Count_4"] = counts[4]
    response_counts_B.loc[column, "Count_5"] = counts[5]

response_counts_B = response_counts_B.astype(int)

response_counts_B

,Count_1,Count_2,Count_3,Count_4,Count_5
PEdS1,4,20,56,203,85
PEdS2,7,7,57,186,111
PEdS3,6,5,57,162,138
PEdS4,2,9,42,191,124
PEmS1,5,14,80,163,106
PEmS2,6,5,64,163,130
PEmS3,4,5,44,187,128
PEmS4,3,3,50,179,133
PU1,3,8,69,178,110
PU2,4,11,76,166,111


In [11]:
# در این سلول متن واقعی هر سؤال پرسشنامه Dataset B به کد فیلد مربوط به آن متصل می‌شود.

question_text_B = {

    "PEdS1": "When I use e-learning, my peers provide information, suggestions, and guidance.",
    "PEdS2": "When I use e-learning, my teacher provides information and helps me improve efficiency.",
    "PEdS3": "When I have questions or doubts about e-learning, my teacher assists me.",
    "PEdS4": "When I encounter difficulties in e-learning, I can always seek help from my peers.",

    "PEmS1": "When I use e-learning, my peers encourage and praise me.",
    "PEmS2": "When I face challenges in e-learning, my teacher is willing to listen and provide the emotional support I need.",
    "PEmS3": "My good friend kindly tells me the truth about how I perform.",
    "PEmS4": "My teacher kindly informed me about my genuine performance.",

    "PU1": "I believe that using e-learning can improve my academic performance.",
    "PU2": "I think that using e-learning can enhance my study efficiency.",
    "PU3": "I feel that using e-learning makes it easy to transform learning materials into concrete knowledge.",

    "CON1": "My experience with e-learning exceeded my expectations.",
    "CON2": "The services provided by e-learning exceeded my expectations.",
    "CON3": "Most of my expectations for e-learning have been confirmed.",

    "SAT1": "I would recommend e-learning to others.",
    "SAT2": "My decision to choose e-learning was right.",
    "SAT3": "I am very satisfied with the use of the electronic learning system.",

    "CI1": "I will use e-learning regularly in the future.",
    "CI2": "I will use e-learning frequently in the future.",
    "CI3": "I will use e-learning more and more in the future."
}

response_counts_B.insert(
    0,
    "Question",
    response_counts_B.index.map(question_text_B)
)

response_counts_B

,Question,Count_1,Count_2,Count_3,Count_4,Count_5
PEdS1,"When I use e-learning, my peers provide inform...",4,20,56,203,85
PEdS2,"When I use e-learning, my teacher provides inf...",7,7,57,186,111
PEdS3,When I have questions or doubts about e-learni...,6,5,57,162,138
PEdS4,"When I encounter difficulties in e-learning, I...",2,9,42,191,124
PEmS1,"When I use e-learning, my peers encourage and ...",5,14,80,163,106
PEmS2,"When I face challenges in e-learning, my teach...",6,5,64,163,130
PEmS3,My good friend kindly tells me the truth about...,4,5,44,187,128
PEmS4,My teacher kindly informed me about my genuine...,3,3,50,179,133
PU1,I believe that using e-learning can improve my...,3,8,69,178,110
PU2,I think that using e-learning can enhance my s...,4,11,76,166,111


In [12]:
# در این سلول برای هر سؤال، درصد پاسخ‌های منفی، خنثی و مثبت و همچنین میانگین و انحراف معیار محاسبه می‌شود.

total_responses_B = len(df_B)

response_counts_B["Negative_%"] = (
    (response_counts_B["Count_1"] + response_counts_B["Count_2"])
    / total_responses_B * 100
).round(2)

response_counts_B["Neutral_%"] = (
    response_counts_B["Count_3"]
    / total_responses_B * 100
).round(2)

response_counts_B["Positive_%"] = (
    (response_counts_B["Count_4"] + response_counts_B["Count_5"])
    / total_responses_B * 100
).round(2)

response_counts_B["Mean"] = [
    round(df_B[column].mean(), 2)
    for column in response_counts_B.index
]

response_counts_B["Std"] = [
    round(df_B[column].std(), 2)
    for column in response_counts_B.index
]

response_counts_B

,Question,Count_1,Count_2,Count_3,Count_4,Count_5,Negative_%,Neutral_%,Positive_%,Mean,Std
PEdS1,"When I use e-learning, my peers provide inform...",4,20,56,203,85,6.52,15.22,78.26,3.94,0.83
PEdS2,"When I use e-learning, my teacher provides inf...",7,7,57,186,111,3.80,15.49,80.71,4.05,0.84
PEdS3,When I have questions or doubts about e-learni...,6,5,57,162,138,2.99,15.49,81.52,4.14,0.84
PEdS4,"When I encounter difficulties in e-learning, I...",2,9,42,191,124,2.99,11.41,85.60,4.16,0.76
PEmS1,"When I use e-learning, my peers encourage and ...",5,14,80,163,106,5.16,21.74,73.10,3.95,0.88
PEmS2,"When I face challenges in e-learning, my teach...",6,5,64,163,130,2.99,17.39,79.62,4.10,0.85
PEmS3,My good friend kindly tells me the truth about...,4,5,44,187,128,2.45,11.96,85.60,4.17,0.77
PEmS4,My teacher kindly informed me about my genuine...,3,3,50,179,133,1.63,13.59,84.78,4.18,0.76
PU1,I believe that using e-learning can improve my...,3,8,69,178,110,2.99,18.75,78.26,4.04,0.80
PU2,I think that using e-learning can enhance my s...,4,11,76,166,111,4.08,20.65,75.27,4.00,0.85


In [13]:
# در این سلول تابعی برای محاسبه پایایی درونی آیتم‌های هر سازه با Cronbach's Alpha تعریف می‌شود.

def cronbach_alpha(items):
    items = items.dropna()

    n_items = items.shape[1]

    item_variances = items.var(axis=0, ddof=1)

    total_score = items.sum(axis=1)

    total_variance = total_score.var(ddof=1)

    alpha = (n_items / (n_items - 1)) * (
        1 - item_variances.sum() / total_variance
    )

    return alpha

In [14]:
# در این سلول مقدار Cronbach's Alpha برای تمام سازه‌های پرسشنامه Dataset B محاسبه می‌شود.

alpha_B = {
    "PEdS": cronbach_alpha(df_B[PEdS_columns_B]),
    "PEmS": cronbach_alpha(df_B[PEmS_columns_B]),
    "PU": cronbach_alpha(df_B[PU_columns_B]),
    "CON": cronbach_alpha(df_B[CON_columns_B]),
    "SAT": cronbach_alpha(df_B[SAT_columns_B]),
    "CI": cronbach_alpha(df_B[CI_columns_B])
}

for construct, alpha in alpha_B.items():
    print(construct, ":", round(alpha, 3))

PEdS : 0.828
PEmS : 0.876
PU : 0.911
CON : 0.905
SAT : 0.911
CI : 0.915


In [15]:
# در این سلول یک نسخه پردازش‌شده از Dataset B ساخته می‌شود و امتیاز میانگین هر سازه برای هر دانشجو محاسبه می‌شود.

df_B_processed = df_B.copy()

df_B_processed["PEdS_score"] = df_B_processed[PEdS_columns_B].mean(axis=1)

df_B_processed["PEmS_score"] = df_B_processed[PEmS_columns_B].mean(axis=1)

df_B_processed["PU_score"] = df_B_processed[PU_columns_B].mean(axis=1)

df_B_processed["CON_score"] = df_B_processed[CON_columns_B].mean(axis=1)

df_B_processed["SAT_score"] = df_B_processed[SAT_columns_B].mean(axis=1)

df_B_processed["CI_score"] = df_B_processed[CI_columns_B].mean(axis=1)

df_B_processed[
    [
        "PEdS_score",
        "PEmS_score",
        "PU_score",
        "CON_score",
        "SAT_score",
        "CI_score"
    ]
].head()

,PEdS_score,PEmS_score,PU_score,CON_score,SAT_score,CI_score
0,4.75,4.25,3.666667,4.333333,4.666667,5.000000
1,4.00,4.00,3.666667,4.333333,4.333333,3.000000
2,3.75,4.00,2.000000,3.333333,2.666667,2.666667
3,3.50,4.50,4.000000,4.666667,5.000000,4.666667
4,3.00,3.25,3.333333,2.000000,3.000000,3.333333


In [16]:
# در این سلول دامنه و آمار توصیفی امتیازهای سازه‌ای Dataset B بررسی می‌شود.

score_columns_B = [
    "PEdS_score",
    "PEmS_score",
    "PU_score",
    "CON_score",
    "SAT_score",
    "CI_score"
]

summary_scores_B = df_B_processed[score_columns_B].describe().T

summary_scores_B

,count,mean,std,min,25%,50%,75%,max
PEdS_score,368.0,4.072690,0.665658,1.25,3.750000,4.0,4.500000,5.0
PEmS_score,368.0,4.102582,0.696801,1.00,3.750000,4.0,4.750000,5.0
PU_score,368.0,4.000906,0.778339,1.00,3.666667,4.0,4.666667,5.0
CON_score,368.0,3.822464,0.856598,1.00,3.333333,4.0,4.333333,5.0
SAT_score,368.0,3.940217,0.797393,1.00,3.583333,4.0,4.416667,5.0
CI_score,368.0,3.959239,0.781542,1.00,3.333333,4.0,4.666667,5.0


In [17]:
# در این سلول آمار توصیفی و مقدار Cronbach's Alpha سازه‌های Dataset B در یک جدول خلاصه ترکیب می‌شوند.

construct_summary_B = pd.DataFrame({
    "Construct": [
        "PEdS",
        "PEmS",
        "PU",
        "CON",
        "SAT",
        "CI"
    ],

    "Mean": [
        df_B_processed["PEdS_score"].mean(),
        df_B_processed["PEmS_score"].mean(),
        df_B_processed["PU_score"].mean(),
        df_B_processed["CON_score"].mean(),
        df_B_processed["SAT_score"].mean(),
        df_B_processed["CI_score"].mean()
    ],

    "Std": [
        df_B_processed["PEdS_score"].std(),
        df_B_processed["PEmS_score"].std(),
        df_B_processed["PU_score"].std(),
        df_B_processed["CON_score"].std(),
        df_B_processed["SAT_score"].std(),
        df_B_processed["CI_score"].std()
    ],

    "Min": [
        df_B_processed["PEdS_score"].min(),
        df_B_processed["PEmS_score"].min(),
        df_B_processed["PU_score"].min(),
        df_B_processed["CON_score"].min(),
        df_B_processed["SAT_score"].min(),
        df_B_processed["CI_score"].min()
    ],

    "Max": [
        df_B_processed["PEdS_score"].max(),
        df_B_processed["PEmS_score"].max(),
        df_B_processed["PU_score"].max(),
        df_B_processed["CON_score"].max(),
        df_B_processed["SAT_score"].max(),
        df_B_processed["CI_score"].max()
    ],

    "Cronbach_Alpha": [
        alpha_B["PEdS"],
        alpha_B["PEmS"],
        alpha_B["PU"],
        alpha_B["CON"],
        alpha_B["SAT"],
        alpha_B["CI"]
    ]
})

construct_summary_B = construct_summary_B.round(3)

construct_summary_B

,Construct,Mean,Std,Min,Max,Cronbach_Alpha
0,PEdS,4.073,0.666,1.25,5.0,0.828
1,PEmS,4.103,0.697,1.00,5.0,0.876
2,PU,4.001,0.778,1.00,5.0,0.911
3,CON,3.822,0.857,1.00,5.0,0.905
4,SAT,3.940,0.797,1.00,5.0,0.911
5,CI,3.959,0.782,1.00,5.0,0.915


In [18]:
# در این سلول نسخه پردازش‌شده Dataset B و جدول خلاصه سازه‌ها ذخیره می‌شوند.

os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(TABLES_FOLDER, exist_ok=True)

df_B_processed.to_csv(
    os.path.join(PROCESSED_FOLDER, "dataset_B_processed.csv"),
    index=False
)

construct_summary_B.to_excel(
    os.path.join(TABLES_FOLDER, "Dataset_B_Construct_Summary.xlsx"),
    index=False
)

print("Dataset B processed file saved successfully.")
print("Dataset B construct summary saved successfully.")


Dataset B processed file saved successfully.
Dataset B construct summary saved successfully.


In [19]:
# در این سلول فایل‌های Dataset C مستقیماً از پوشه excluded_datasets نمایش داده می‌شوند.

files_C = os.listdir(DATASET_C_FOLDER)

for file in files_C:
    print(file)


dataset_c.docx
Table 1_Understanding continuance intention towards e-learning platforms_ a structural model based on expectation confirmation and individual innovati.xlsx


In [20]:
# در این سلول فایل خام Dataset C مستقیماً از پوشه excluded_datasets خوانده می‌شود.

file_C = os.path.join(DATASET_C_FOLDER, 'Table 1_Understanding continuance intention towards e-learning platforms_ a structural model based on expectation confirmation and individual innovati.xlsx')
df_C = pd.read_excel(file_C)
df_C.head()


,Gen,Age,Edu Qu,Fi of St,Pr Exp,Cou Com,Swa,Cou,Ude,IIN 1,...,EC 1,EC 2,EC 3,S 1,S 2,S 3,CI 1,CI 2,CI 3,CI 4
0,2,2,1,2,1,1,1,0,0,5,...,5,4,5,5,5,5,5,5,5,5
1,1,2,2,1,1,1,1,0,0,4,...,4,4,4,4,4,4,4,3,5,4
2,1,1,1,1,1,1,0,1,1,5,...,5,5,5,5,5,5,5,4,5,5
3,2,1,1,1,1,1,0,1,0,5,...,4,4,4,4,4,4,4,4,4,4
4,2,3,1,2,2,1,1,0,0,4,...,4,4,4,4,4,4,4,4,4,4


In [21]:
# در این سلول ابعاد Dataset C و نام تمام ستون‌های آن بررسی می‌شوند.

print("Shape of Dataset C:", df_C.shape)

print("\nColumns of Dataset C:")
for column in df_C.columns:
    print(column)

Shape of Dataset C: (166, 26)

Columns of Dataset C:
Gen
Age
Edu Qu
Fi of St
Pr Exp
Cou Com
Swa
Cou
Ude
IIN 1
IIN 2
IIN 3
PU 1
PU 2
PU 3
PU 4
EC 1
EC 2
EC 3
S 1
S 2
S 3
CI 1
CI 2
CI 3
CI 4


In [22]:
# در این سلول ستون‌های Dataset C بر اساس اطلاعات عمومی و سازه‌های پرسشنامه دسته‌بندی می‌شوند.

general_columns_C = [
    "Gen",
    "Age",
    "Edu Qu",
    "Fi of St",
    "Pr Exp",
    "Cou Com",
    "Swa",
    "Cou",
    "Ude"
]

IIN_columns_C = [
    "IIN 1",
    "IIN 2",
    "IIN 3"
]

PU_columns_C = [
    "PU 1",
    "PU 2",
    "PU 3",
    "PU 4"
]

EC_columns_C = [
    "EC 1",
    "EC 2",
    "EC 3"
]

S_columns_C = [
    "S 1",
    "S 2",
    "S 3"
]

CI_columns_C = [
    "CI 1",
    "CI 2",
    "CI 3",
    "CI 4"
]

print("General columns:", general_columns_C)
print("IIN columns:", IIN_columns_C)
print("PU columns:", PU_columns_C)
print("EC columns:", EC_columns_C)
print("Satisfaction columns:", S_columns_C)
print("CI columns:", CI_columns_C)

General columns: ['Gen', 'Age', 'Edu Qu', 'Fi of St', 'Pr Exp', 'Cou Com', 'Swa', 'Cou', 'Ude']
IIN columns: ['IIN 1', 'IIN 2', 'IIN 3']
PU columns: ['PU 1', 'PU 2', 'PU 3', 'PU 4']
EC columns: ['EC 1', 'EC 2', 'EC 3']
Satisfaction columns: ['S 1', 'S 2', 'S 3']
CI columns: ['CI 1', 'CI 2', 'CI 3', 'CI 4']


In [23]:
# در این سلول نوع داده هر ستون در Dataset C بررسی می‌شود.

df_C.dtypes

Gen         int64
Age         int64
Edu Qu      int64
Fi of St    int64
Pr Exp      int64
Cou Com     int64
Swa         int64
Cou         int64
Ude         int64
IIN 1       int64
IIN 2       int64
IIN 3       int64
PU 1        int64
PU 2        int64
PU 3        int64
PU 4        int64
EC 1        int64
EC 2        int64
EC 3        int64
S 1         int64
S 2         int64
S 3         int64
CI 1        int64
CI 2        int64
CI 3        int64
CI 4        int64
dtype: object

In [24]:
# در این سلول تعداد مقادیر گمشده در هر ستون Dataset C بررسی می‌شود.

missing_C = df_C.isnull().sum()

print(missing_C)

Gen         0
Age         0
Edu Qu      0
Fi of St    0
Pr Exp      0
Cou Com     0
Swa         0
Cou         0
Ude         0
IIN 1       0
IIN 2       0
IIN 3       0
PU 1        0
PU 2        0
PU 3        0
PU 4        0
EC 1        0
EC 2        0
EC 3        0
S 1         0
S 2         0
S 3         0
CI 1        0
CI 2        0
CI 3        0
CI 4        0
dtype: int64


In [25]:
# در این سلول بررسی می‌شود که آیا ردیف کاملاً تکراری در Dataset C وجود دارد یا خیر.

duplicate_rows_C = df_C.duplicated().sum()

print("Number of completely duplicated rows:", duplicate_rows_C)

Number of completely duplicated rows: 123


In [26]:
# در این سلول تعداد رکوردهای یکتا و تعداد دفعات تکرار هر الگوی کامل در Dataset C بررسی می‌شود.

total_rows_C = len(df_C)

unique_rows_C = len(df_C.drop_duplicates())

duplicate_rows_C = df_C.duplicated().sum()

print("Total rows:", total_rows_C)
print("Unique complete rows:", unique_rows_C)
print("Duplicated rows:", duplicate_rows_C)


repeat_patterns_C = (
    df_C.value_counts()
    .reset_index(name="Repeat_Count")
    .sort_values("Repeat_Count", ascending=False)
)

repeat_patterns_C.head(15)

Total rows: 166
Unique complete rows: 43
Duplicated rows: 123


,Gen,Age,Edu Qu,Fi of St,Pr Exp,Cou Com,Swa,Cou,Ude,IIN 1,...,EC 2,EC 3,S 1,S 2,S 3,CI 1,CI 2,CI 3,CI 4,Repeat_Count
0,1,3,2,2,1,1,1,0,0,4,...,5,5,5,5,5,5,5,4,4,6
2,2,3,1,3,1,1,1,1,1,4,...,3,3,3,2,2,3,3,3,3,6
3,2,3,1,1,2,1,1,0,0,4,...,3,3,3,3,3,3,3,3,3,6
4,2,2,1,1,2,1,1,0,0,4,...,4,4,4,4,5,5,4,4,4,6
5,1,3,2,2,1,1,0,0,1,5,...,3,4,5,4,4,5,4,5,4,6
6,1,3,1,3,1,1,0,1,0,3,...,5,5,5,5,5,5,5,5,5,6
7,1,2,2,2,3,2,0,1,0,3,...,3,4,4,3,3,3,3,3,3,6
8,2,3,2,2,2,2,1,0,0,3,...,4,4,4,4,4,3,3,3,4,6
1,1,2,2,2,1,1,0,1,0,4,...,5,4,4,4,4,5,5,4,4,6
16,2,2,1,2,1,1,1,0,0,5,...,4,5,5,5,5,5,5,5,5,5


### Data Quality Decision – Dataset C

Dataset C initially contained 166 observations.

A full-row duplicate analysis showed:

- Total observations: 166
- Unique complete observations: 43
- Duplicated observations: 123

Several complete response patterns were repeated multiple times, including identical
demographic characteristics and questionnaire responses across all 26 variables.

Because the available documentation does not explain this extensive duplication,
Dataset C was not used for external validation.

The original raw file was retained without modification for reproducibility.

In [28]:
# در این سلول فایل خام EFL از پوشه datasets_EFL خوانده می‌شود.

file_EFL = os.path.join(DATASET_EFL_FOLDER, "Raw Data_Investigating the Predictive Factors Influencing EFL Students' Continuance Intention to Use E-Learning.sav")
df_EFL = pd.read_spss(file_EFL)
df_EFL.head()


,Gender,age,Grades,parentseducation,monthlyincome,PE1,PE2,PE3,PE4,PE5,...,CI3,CON1,CON2,CON3,PE,PU,PEOU,SAT,CI,CON
0,2.0,2.0,1.0,3.0,2.0,5.0,5.0,5.0,5.0,5.0,...,5.0,4.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,4.333333
1,1.0,2.0,1.0,1.0,1.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.000000
2,1.0,2.0,1.0,1.0,2.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.000000
3,1.0,2.0,1.0,2.0,1.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.000000
4,1.0,2.0,1.0,1.0,2.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.000000


In [29]:
# در این سلول تعداد کل رکوردها، تعداد ردیف‌های یکتا و تعداد ردیف‌های کاملاً تکراری در دیتاست EFL بررسی می‌شود.

total_rows_EFL = len(df_EFL)

unique_rows_EFL = len(df_EFL.drop_duplicates())

duplicate_rows_EFL = df_EFL.duplicated().sum()

print("Shape of EFL Dataset:", df_EFL.shape)
print("Total rows:", total_rows_EFL)
print("Unique complete rows:", unique_rows_EFL)
print("Duplicated rows:", duplicate_rows_EFL)

Shape of EFL Dataset: (435, 32)
Total rows: 435
Unique complete rows: 330
Duplicated rows: 105


In [30]:
# در این سلول بررسی می‌شود رکوردهای تکراری EFL چند بار تکرار شده‌اند
# و آیا تکرارها ناشی از پاسخ‌های کاملاً یکسان پرسشنامه هستند یا خیر.

repeat_patterns_EFL = (
    df_EFL.value_counts()
    .reset_index(name="Repeat_Count")
    .sort_values("Repeat_Count", ascending=False)
)

print("Top repeated complete rows:")
display(repeat_patterns_EFL.head(15))

Top repeated complete rows:


,Gender,age,Grades,parentseducation,monthlyincome,PE1,PE2,PE3,PE4,PE5,...,CON1,CON2,CON3,PE,PU,PEOU,SAT,CI,CON,Repeat_Count
0,2.0,2.0,1.0,2.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,14
2,2.0,2.0,1.0,3.0,2.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,10
1,2.0,2.0,1.0,1.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,10
3,2.0,2.0,1.0,2.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,3.0,4.000000,4.000000,4.000000,4.000000,4.000000,3.666667,5
9,2.0,2.0,1.0,2.0,3.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,3.333333,4
13,2.0,2.0,1.0,4.0,3.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4
12,2.0,1.0,1.0,3.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4
11,2.0,2.0,1.0,3.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4
10,1.0,2.0,1.0,2.0,2.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,3.666667,4.000000,4.000000,3.666667,4.000000,4
8,2.0,2.0,1.0,1.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.000000,3.333333,4.000000,4.000000,4.000000,4.000000,4


In [31]:
# در این سلول تعداد الگوهای یکتای پاسخ به آیتم‌های پرسشنامه EFL بررسی می‌شود.

questionnaire_columns_EFL = [
    "PE1", "PE2", "PE3", "PE4", "PE5", "PE6",
    "PU1", "PU2", "PU3",
    "PEOU1", "PEOU2", "PEOU3",
    "SAT1", "SAT2", "SAT3",
    "CI1", "CI2", "CI3",
    "CON1", "CON2", "CON3"
]

unique_questionnaire_EFL = (
    df_EFL[questionnaire_columns_EFL]
    .drop_duplicates()
    .shape[0]
)

duplicate_questionnaire_EFL = (
    df_EFL[questionnaire_columns_EFL]
    .duplicated()
    .sum()
)

print("Unique questionnaire response patterns:", unique_questionnaire_EFL)
print("Duplicated questionnaire response patterns:", duplicate_questionnaire_EFL)

Unique questionnaire response patterns: 279
Duplicated questionnaire response patterns: 156


In [32]:
# در این سلول پاسخ‌دهندگانی شناسایی می‌شوند که به تمام آیتم‌های پرسشنامه یک امتیاز یکسان داده‌اند.

straightline_mask_EFL = (
    df_EFL[questionnaire_columns_EFL]
    .nunique(axis=1)
    == 1
)

straightline_count_EFL = straightline_mask_EFL.sum()

print("Number of straight-line responses:", straightline_count_EFL)

df_EFL.loc[
    straightline_mask_EFL,
    questionnaire_columns_EFL
].head(20)

Number of straight-line responses: 108


,PE1,PE2,PE3,PE4,PE5,PE6,PU1,PU2,PU3,PEOU1,...,PEOU3,SAT1,SAT2,SAT3,CI1,CI2,CI3,CON1,CON2,CON3
1,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
2,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
3,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
4,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
5,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
89,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
91,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
93,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
96,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
97,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0


### Data Quality Decision – EFL Dataset

The EFL dataset contained 435 observations.

Data-quality screening identified:

- 435 total observations
- 330 unique complete rows
- 105 duplicated complete rows
- 279 unique questionnaire response patterns
- extensive repeated questionnaire-response patterns
- multiple straight-line responses in which the same Likert score was selected
  across all questionnaire items

Because these patterns raise concerns about the independence and quality of
observations, the dataset was not selected for external validation.

The original raw dataset was retained without modification for reproducibility.

In [33]:
# در این سلول یک نسخه جداگانه از EFL ساخته می‌شود و فقط ردیف‌های کاملاً تکراری از نسخه پردازش‌شده حذف می‌شوند؛ داده خام بدون تغییر باقی می‌ماند.

df_EFL_deduplicated = df_EFL.drop_duplicates().copy()

print("Original rows:", len(df_EFL))
print("Rows after exact duplicate removal:", len(df_EFL_deduplicated))
print("Removed exact duplicates:", len(df_EFL) - len(df_EFL_deduplicated))

Original rows: 435
Rows after exact duplicate removal: 330
Removed exact duplicates: 105


In [34]:
# در این سلول تعداد پاسخ‌های Straight-line در نسخه بدون Duplicate دیتاست EFL بررسی می‌شود.

straightline_mask_EFL_clean = (
    df_EFL_deduplicated[questionnaire_columns_EFL]
    .nunique(axis=1)
    == 1
)

straightline_count_EFL_clean = straightline_mask_EFL_clean.sum()

print("Rows after exact duplicate removal:", len(df_EFL_deduplicated))
print("Straight-line responses:", straightline_count_EFL_clean)
print(
    "Percentage of straight-line responses:",
    round(straightline_count_EFL_clean / len(df_EFL_deduplicated) * 100, 2),
    "%"
)

Rows after exact duplicate removal: 330
Straight-line responses: 45
Percentage of straight-line responses: 13.64 %


In [35]:
# در این سلول نوع امتیاز Straight-lineها بررسی می‌شود؛ مثلاً تمام پاسخ‌ها 4 یا تمام پاسخ‌ها 5 بوده‌اند.

straightline_rows_EFL = df_EFL_deduplicated.loc[
    straightline_mask_EFL_clean,
    questionnaire_columns_EFL
]

straightline_rows_EFL.iloc[:, 0].value_counts().sort_index()

PE1
3.0     6
4.0    36
5.0     3
Name: count, dtype: int64

In [36]:
# در این سلول پاسخ‌های Straight-line از نسخه بدون Duplicate حذف می‌شوند و دیتاست پاک‌سازی‌شده EFL ساخته می‌شود.

df_EFL_clean = df_EFL_deduplicated.loc[
    ~straightline_mask_EFL_clean
].copy()

print("Original raw rows:", len(df_EFL))
print("After exact duplicate removal:", len(df_EFL_deduplicated))
print("After straight-line removal:", len(df_EFL_clean))

print(
    "Total removed:",
    len(df_EFL) - len(df_EFL_clean)
)

Original raw rows: 435
After exact duplicate removal: 330
After straight-line removal: 285
Total removed: 150


In [37]:
# در این سلول بررسی می‌شود که در نسخه پاک‌سازی‌شده EFL هیچ Duplicate کامل یا پاسخ Straight-line باقی نمانده باشد.

remaining_duplicates_EFL = df_EFL_clean.duplicated().sum()

remaining_straightlines_EFL = (
    df_EFL_clean[questionnaire_columns_EFL]
    .nunique(axis=1)
    .eq(1)
    .sum()
)

print("Final cleaned rows:", len(df_EFL_clean))
print("Remaining complete duplicates:", remaining_duplicates_EFL)
print("Remaining straight-line responses:", remaining_straightlines_EFL)

Final cleaned rows: 285
Remaining complete duplicates: 0
Remaining straight-line responses: 0


In [38]:
# در این سلول نام تمام ستون‌های دیتاست پاک‌سازی‌شده EFL بررسی می‌شود تا آیتم‌های خام و امتیازهای ترکیبی از هم تفکیک شوند.

print("Shape of cleaned EFL dataset:", df_EFL_clean.shape)

print("\nColumns of cleaned EFL dataset:")

for column in df_EFL_clean.columns:
    print(column)

Shape of cleaned EFL dataset: (285, 32)

Columns of cleaned EFL dataset:
Gender
age
Grades
parentseducation
monthlyincome
PE1
PE2
PE3
PE4
PE5
PE6
PU1
PU2
PU3
PEOU1
PEOU2
PEOU3
SAT1
SAT2
SAT3
CI1
CI2
CI3
CON1
CON2
CON3
PE
PU
PEOU
SAT
CI
CON


In [39]:
# در این سلول ستون‌های EFL به متغیرهای جمعیت‌شناختی، آیتم‌های خام پرسشنامه و امتیازهای ترکیبی تقسیم می‌شوند.

demographic_columns_EFL = [
    "Gender",
    "age",
    "Grades",
    "parentseducation",
    "monthlyincome"
]

PE_columns_EFL = [
    "PE1", "PE2", "PE3", "PE4", "PE5", "PE6"
]

PU_columns_EFL = [
    "PU1", "PU2", "PU3"
]

PEOU_columns_EFL = [
    "PEOU1", "PEOU2", "PEOU3"
]

SAT_columns_EFL = [
    "SAT1", "SAT2", "SAT3"
]

CI_columns_EFL = [
    "CI1", "CI2", "CI3"
]

CON_columns_EFL = [
    "CON1", "CON2", "CON3"
]

composite_columns_EFL = [
    "PE",
    "PU",
    "PEOU",
    "SAT",
    "CI",
    "CON"
]

print("Demographic columns:", demographic_columns_EFL)
print("PE columns:", PE_columns_EFL)
print("PU columns:", PU_columns_EFL)
print("PEOU columns:", PEOU_columns_EFL)
print("SAT columns:", SAT_columns_EFL)
print("CI columns:", CI_columns_EFL)
print("CON columns:", CON_columns_EFL)
print("Existing composite columns:", composite_columns_EFL)

Demographic columns: ['Gender', 'age', 'Grades', 'parentseducation', 'monthlyincome']
PE columns: ['PE1', 'PE2', 'PE3', 'PE4', 'PE5', 'PE6']
PU columns: ['PU1', 'PU2', 'PU3']
PEOU columns: ['PEOU1', 'PEOU2', 'PEOU3']
SAT columns: ['SAT1', 'SAT2', 'SAT3']
CI columns: ['CI1', 'CI2', 'CI3']
CON columns: ['CON1', 'CON2', 'CON3']
Existing composite columns: ['PE', 'PU', 'PEOU', 'SAT', 'CI', 'CON']


In [40]:
# در این سلول تعداد مقادیر گمشده در هر ستون نسخه پاک‌سازی‌شده EFL بررسی می‌شود.

missing_EFL = df_EFL_clean.isnull().sum()

print(missing_EFL)

Gender              0
age                 0
Grades              0
parentseducation    0
monthlyincome       0
PE1                 0
PE2                 0
PE3                 0
PE4                 0
PE5                 0
PE6                 0
PU1                 0
PU2                 0
PU3                 0
PEOU1               0
PEOU2               0
PEOU3               0
SAT1                0
SAT2                0
SAT3                0
CI1                 0
CI2                 0
CI3                 0
CON1                0
CON2                0
CON3                0
PE                  0
PU                  0
PEOU                0
SAT                 0
CI                  0
CON                 0
dtype: int64


In [41]:
# در این سلول مقادیر یکتای هر آیتم پرسشنامه EFL بررسی می‌شود تا مطمئن شویم همه پاسخ‌ها در دامنه معتبر 1 تا 5 هستند.

questionnaire_columns_EFL = (
    PE_columns_EFL
    + PU_columns_EFL
    + PEOU_columns_EFL
    + SAT_columns_EFL
    + CI_columns_EFL
    + CON_columns_EFL
)

for column in questionnaire_columns_EFL:
    print(column, ":", sorted(df_EFL_clean[column].unique()))

PE1 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PE2 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PE3 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PE4 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PE5 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PE6 : [np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PU1 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PU2 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PU3 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PEOU1 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PEOU2 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
P

In [42]:
# در این سلول پایایی درونی سازه‌های پرسشنامه در نسخه پاک‌سازی‌شده EFL با Cronbach's Alpha محاسبه می‌شود.

alpha_EFL = {
    "PE": cronbach_alpha(df_EFL_clean[PE_columns_EFL]),
    "PU": cronbach_alpha(df_EFL_clean[PU_columns_EFL]),
    "PEOU": cronbach_alpha(df_EFL_clean[PEOU_columns_EFL]),
    "SAT": cronbach_alpha(df_EFL_clean[SAT_columns_EFL]),
    "CI": cronbach_alpha(df_EFL_clean[CI_columns_EFL]),
    "CON": cronbach_alpha(df_EFL_clean[CON_columns_EFL])
}

for construct, alpha in alpha_EFL.items():
    print(construct, ":", round(alpha, 3))

PE : 0.896
PU : 0.802
PEOU : 0.737
SAT : 0.78
CI : 0.836
CON : 0.821


In [43]:
# در این سلول امتیاز میانگین هر سازه از روی آیتم‌های خام پرسشنامه EFL محاسبه می‌شود.

df_EFL_clean["PE_score_calc"] = df_EFL_clean[PE_columns_EFL].mean(axis=1)
df_EFL_clean["PU_score_calc"] = df_EFL_clean[PU_columns_EFL].mean(axis=1)
df_EFL_clean["PEOU_score_calc"] = df_EFL_clean[PEOU_columns_EFL].mean(axis=1)
df_EFL_clean["SAT_score_calc"] = df_EFL_clean[SAT_columns_EFL].mean(axis=1)
df_EFL_clean["CI_score_calc"] = df_EFL_clean[CI_columns_EFL].mean(axis=1)
df_EFL_clean["CON_score_calc"] = df_EFL_clean[CON_columns_EFL].mean(axis=1)

df_EFL_clean[
    [
        "PE", "PE_score_calc",
        "PU", "PU_score_calc",
        "PEOU", "PEOU_score_calc",
        "SAT", "SAT_score_calc",
        "CI", "CI_score_calc",
        "CON", "CON_score_calc"
    ]
].head()

,PE,PE_score_calc,PU,PU_score_calc,PEOU,PEOU_score_calc,SAT,SAT_score_calc,CI,CI_score_calc,CON,CON_score_calc
0,5.000000,5.000000,5.0,5.0,5.0,5.0,5.000000,5.000000,5.000000,5.000000,4.333333,4.333333
6,4.500000,4.500000,5.0,5.0,5.0,5.0,4.666667,4.666667,5.000000,5.000000,5.000000,5.000000
7,4.000000,4.000000,5.0,5.0,5.0,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
8,3.833333,3.833333,5.0,5.0,5.0,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
9,5.000000,5.000000,5.0,5.0,5.0,5.0,5.000000,5.000000,4.333333,4.333333,5.000000,5.000000


In [44]:
# در این سلول حداکثر اختلاف بین Compositeهای اصلی فایل و Compositeهای محاسبه‌شده توسط خودمان در تمام 285 رکورد بررسی می‌شود.

comparison_EFL = {
    "PE": (df_EFL_clean["PE"] - df_EFL_clean["PE_score_calc"]).abs().max(),
    "PU": (df_EFL_clean["PU"] - df_EFL_clean["PU_score_calc"]).abs().max(),
    "PEOU": (df_EFL_clean["PEOU"] - df_EFL_clean["PEOU_score_calc"]).abs().max(),
    "SAT": (df_EFL_clean["SAT"] - df_EFL_clean["SAT_score_calc"]).abs().max(),
    "CI": (df_EFL_clean["CI"] - df_EFL_clean["CI_score_calc"]).abs().max(),
    "CON": (df_EFL_clean["CON"] - df_EFL_clean["CON_score_calc"]).abs().max()
}

for construct, difference in comparison_EFL.items():
    print(construct, "Maximum difference:", difference)

PE Maximum difference: 0.0
PU Maximum difference: 0.0
PEOU Maximum difference: 0.0
SAT Maximum difference: 0.0
CI Maximum difference: 0.0
CON Maximum difference: 0.0


In [45]:
# در این سلول Compositeهای آماده فایل حذف می‌شوند و Compositeهای محاسبه‌شده توسط خودمان با نام‌های استاندارد جایگزین می‌شوند.

df_EFL_clean = df_EFL_clean.drop(
    columns=[
        "PE",
        "PU",
        "PEOU",
        "SAT",
        "CI",
        "CON"
    ]
)

df_EFL_clean = df_EFL_clean.rename(
    columns={
        "PE_score_calc": "PE_score",
        "PU_score_calc": "PU_score",
        "PEOU_score_calc": "PEOU_score",
        "SAT_score_calc": "SAT_score",
        "CI_score_calc": "CI_score",
        "CON_score_calc": "CON_score"
    }
)

print("Shape after removing old composite columns:", df_EFL_clean.shape)

print("\nFinal composite columns:")
print([
    "PE_score",
    "PU_score",
    "PEOU_score",
    "SAT_score",
    "CI_score",
    "CON_score"
])

Shape after removing old composite columns: (285, 32)

Final composite columns:
['PE_score', 'PU_score', 'PEOU_score', 'SAT_score', 'CI_score', 'CON_score']


In [46]:
# در این سلول آمار توصیفی امتیازهای ترکیبی سازه‌ها در دیتاست پاک‌سازی‌شده EFL بررسی می‌شود.

composite_score_columns_EFL = [
    "PE_score",
    "PU_score",
    "PEOU_score",
    "SAT_score",
    "CI_score",
    "CON_score"
]

df_EFL_clean[
    composite_score_columns_EFL
].describe().T

,count,mean,std,min,25%,50%,75%,max
PE_score,285.0,3.875439,0.556282,1.666667,3.666667,4.000000,4.0,5.0
PU_score,285.0,3.775439,0.644178,1.333333,3.333333,4.000000,4.0,5.0
PEOU_score,285.0,3.705263,0.626482,1.333333,3.333333,3.666667,4.0,5.0
SAT_score,285.0,3.790643,0.633128,1.666667,3.333333,4.000000,4.0,5.0
CI_score,285.0,3.837427,0.625866,1.333333,3.666667,4.000000,4.0,5.0
CON_score,285.0,3.727485,0.611737,1.666667,3.333333,4.000000,4.0,5.0


In [47]:
# در این سلول جدول خلاصه سازه‌های EFL شامل میانگین، انحراف معیار، حداقل، حداکثر و Cronbach's Alpha ساخته می‌شود.

construct_summary_EFL = pd.DataFrame({
    "Construct": [
        "PE",
        "PU",
        "PEOU",
        "SAT",
        "CI",
        "CON"
    ],

    "Mean": [
        df_EFL_clean["PE_score"].mean(),
        df_EFL_clean["PU_score"].mean(),
        df_EFL_clean["PEOU_score"].mean(),
        df_EFL_clean["SAT_score"].mean(),
        df_EFL_clean["CI_score"].mean(),
        df_EFL_clean["CON_score"].mean()
    ],

    "Std": [
        df_EFL_clean["PE_score"].std(),
        df_EFL_clean["PU_score"].std(),
        df_EFL_clean["PEOU_score"].std(),
        df_EFL_clean["SAT_score"].std(),
        df_EFL_clean["CI_score"].std(),
        df_EFL_clean["CON_score"].std()
    ],

    "Min": [
        df_EFL_clean["PE_score"].min(),
        df_EFL_clean["PU_score"].min(),
        df_EFL_clean["PEOU_score"].min(),
        df_EFL_clean["SAT_score"].min(),
        df_EFL_clean["CI_score"].min(),
        df_EFL_clean["CON_score"].min()
    ],

    "Max": [
        df_EFL_clean["PE_score"].max(),
        df_EFL_clean["PU_score"].max(),
        df_EFL_clean["PEOU_score"].max(),
        df_EFL_clean["SAT_score"].max(),
        df_EFL_clean["CI_score"].max(),
        df_EFL_clean["CON_score"].max()
    ],

    "Cronbach_Alpha": [
        alpha_EFL["PE"],
        alpha_EFL["PU"],
        alpha_EFL["PEOU"],
        alpha_EFL["SAT"],
        alpha_EFL["CI"],
        alpha_EFL["CON"]
    ]
})

construct_summary_EFL = construct_summary_EFL.round(3)

construct_summary_EFL

,Construct,Mean,Std,Min,Max,Cronbach_Alpha
0,PE,3.875,0.556,1.667,5.0,0.896
1,PU,3.775,0.644,1.333,5.0,0.802
2,PEOU,3.705,0.626,1.333,5.0,0.737
3,SAT,3.791,0.633,1.667,5.0,0.780
4,CI,3.837,0.626,1.333,5.0,0.836
5,CON,3.727,0.612,1.667,5.0,0.821


In [48]:
# در این سلول نسخه فیلترشده EFL برای تحلیل حساسیت و جدول خلاصه سازه‌های آن ذخیره می‌شوند.
# توجه: این نسخه به دلیل حذف duplicate و straight-line، نسخه Primary نیست و برای sensitivity/robustness نگهداری می‌شود.

os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(TABLES_FOLDER, exist_ok=True)

df_EFL_clean.to_csv(
    os.path.join(PROCESSED_FOLDER, "dataset_EFL_sensitivity_filtered.csv"),
    index=False
)

construct_summary_EFL.to_excel(
    os.path.join(TABLES_FOLDER, "Dataset_EFL_Construct_Summary_Filtered.xlsx"),
    index=False
)

print("EFL sensitivity-filtered dataset saved successfully.")
print("EFL filtered construct summary saved successfully.")


EFL sensitivity-filtered dataset saved successfully.
EFL filtered construct summary saved successfully.


In [49]:
# در این سلول برای هر سؤال EFL متن انگلیسی و فارسی، تعداد پاسخ‌های 1 تا 5 و آمار توصیفی محاسبه می‌شود.

question_texts_EFL = {

    "PEOU1": "Engaging with the e-learning platform feels straightforward and intuitive.",
    "PEOU2": "The e-learning platform requires little mental effort from me.",
    "PEOU3": "I find it easy to navigate the e-learning platform for design purposes.",

    "PU1": "I find the e-learning platform highly motivating when engaging in design tasks.",
    "PU2": "The e-learning platform proves to be efficient when managing design-related work.",
    "PU3": "I experience ease when using the e-learning platform for design tasks.",

    "CON1": "My experience with the e-learning platform exceeded what I initially expected.",
    "CON2": "The e-learning platform fulfills needs that go beyond my basic service requirements.",
    "CON3": "The quality of service from the e-learning platform was above what I anticipated.",

    "SAT1": "I am content with the services offered by the e-learning platform.",
    "SAT2": "I would recommend the e-learning platform to other users.",
    "SAT3": "Choosing this e-learning platform was the right decision for me.",

    "PE1": "I enjoy the experience of sharing knowledge on the e-learning platform.",
    "PE2": "Studying English helps me pursue better educational and career prospects abroad.",
    "PE3": "I take pleasure in sharing my knowledge on the e-learning platform.",
    "PE4": "It feels gratifying to help others by sharing knowledge on this platform.",
    "PE5": "My interaction in knowledge sharing with other students feels rewarding.",
    "PE6": "I experience satisfaction when I share knowledge on the e-learning platform.",

    "CI1": "I plan to keep using the e-learning platform long-term.",
    "CI2": "I aim to use the e-learning platform consistently in the future.",
    "CI3": "I foresee myself frequently using the e-learning platform moving forward."
}


question_texts_fa_EFL = {

    "PEOU1": "کار با پلتفرم یادگیری الکترونیکی برای من ساده و قابل‌درک است.",
    "PEOU2": "استفاده از پلتفرم یادگیری الکترونیکی به تلاش ذهنی کمی از سوی من نیاز دارد.",
    "PEOU3": "پیمایش و کار با پلتفرم یادگیری الکترونیکی برای اهداف طراحی را آسان می‌دانم.",

    "PU1": "پلتفرم یادگیری الکترونیکی هنگام انجام وظایف طراحی برای من بسیار انگیزه‌بخش است.",
    "PU2": "پلتفرم یادگیری الکترونیکی در مدیریت کارهای مرتبط با طراحی کارآمد است.",
    "PU3": "استفاده از پلتفرم یادگیری الکترونیکی برای انجام وظایف طراحی را آسان می‌دانم.",

    "CON1": "تجربه من از پلتفرم یادگیری الکترونیکی فراتر از چیزی بود که در ابتدا انتظار داشتم.",
    "CON2": "پلتفرم یادگیری الکترونیکی نیازهایی فراتر از الزامات اولیه خدماتی من را برآورده می‌کند.",
    "CON3": "کیفیت خدمات پلتفرم یادگیری الکترونیکی بالاتر از چیزی بود که انتظار داشتم.",

    "SAT1": "از خدمات ارائه‌شده توسط پلتفرم یادگیری الکترونیکی راضی هستم.",
    "SAT2": "پلتفرم یادگیری الکترونیکی را به سایر کاربران توصیه می‌کنم.",
    "SAT3": "انتخاب این پلتفرم یادگیری الکترونیکی تصمیم درستی برای من بود.",

    "PE1": "از تجربه به‌اشتراک‌گذاری دانش در پلتفرم یادگیری الکترونیکی لذت می‌برم.",
    "PE2": "مطالعه زبان انگلیسی به من کمک می‌کند فرصت‌های آموزشی و شغلی بهتری را در خارج از کشور دنبال کنم.",
    "PE3": "از به‌اشتراک‌گذاری دانش خود در پلتفرم یادگیری الکترونیکی لذت می‌برم.",
    "PE4": "کمک به دیگران از طریق به‌اشتراک‌گذاری دانش در این پلتفرم برای من رضایت‌بخش است.",
    "PE5": "تعامل من با سایر دانشجویان در زمینه به‌اشتراک‌گذاری دانش، تجربه‌ای رضایت‌بخش است.",
    "PE6": "هنگامی که دانش خود را در پلتفرم یادگیری الکترونیکی به اشتراک می‌گذارم، احساس رضایت می‌کنم.",

    "CI1": "قصد دارم در بلندمدت به استفاده از پلتفرم یادگیری الکترونیکی ادامه دهم.",
    "CI2": "قصد دارم در آینده به‌طور منظم از پلتفرم یادگیری الکترونیکی استفاده کنم.",
    "CI3": "پیش‌بینی می‌کنم در آینده به‌طور مکرر از پلتفرم یادگیری الکترونیکی استفاده کنم."
}


construct_map_EFL = {
    "PEOU1": "PEOU", "PEOU2": "PEOU", "PEOU3": "PEOU",
    "PU1": "PU", "PU2": "PU", "PU3": "PU",
    "CON1": "CON", "CON2": "CON", "CON3": "CON",
    "SAT1": "SAT", "SAT2": "SAT", "SAT3": "SAT",
    "PE1": "PE", "PE2": "PE", "PE3": "PE",
    "PE4": "PE", "PE5": "PE", "PE6": "PE",
    "CI1": "CI", "CI2": "CI", "CI3": "CI"
}


questionnaire_response_EFL = []

for column in questionnaire_columns_EFL:

    counts = (
        df_EFL_clean[column]
        .value_counts()
        .reindex([1, 2, 3, 4, 5], fill_value=0)
    )

    total = len(df_EFL_clean)

    questionnaire_response_EFL.append({

        "Construct": construct_map_EFL[column],

        "Item": column,

        "English_Question": question_texts_EFL[column],

        "Persian_Question": question_texts_fa_EFL[column],

        "Count_1": int(counts[1]),
        "Count_2": int(counts[2]),
        "Count_3": int(counts[3]),
        "Count_4": int(counts[4]),
        "Count_5": int(counts[5]),

        "Total": total,

        "Negative_%": round(
            ((counts[1] + counts[2]) / total) * 100, 2
        ),

        "Neutral_%": round(
            (counts[3] / total) * 100, 2
        ),

        "Positive_%": round(
            ((counts[4] + counts[5]) / total) * 100, 2
        ),

        "Mean": round(
            df_EFL_clean[column].mean(), 3
        ),

        "Std": round(
            df_EFL_clean[column].std(), 3
        )
    })


response_table_EFL = pd.DataFrame(questionnaire_response_EFL)

response_table_EFL

,Construct,Item,English_Question,Persian_Question,Count_1,Count_2,Count_3,Count_4,Count_5,Total,Negative_%,Neutral_%,Positive_%,Mean,Std
0,PE,PE1,I enjoy the experience of sharing knowledge on...,از تجربه به‌اشتراک‌گذاری دانش در پلتفرم یادگیر...,2,2,73,165,43,285,1.40,25.61,72.98,3.860,0.693
1,PE,PE2,Studying English helps me pursue better educat...,مطالعه زبان انگلیسی به من کمک می‌کند فرصت‌های ...,2,9,70,162,42,285,3.86,24.56,71.58,3.818,0.742
2,PE,PE3,I take pleasure in sharing my knowledge on the...,از به‌اشتراک‌گذاری دانش خود در پلتفرم یادگیری ...,1,5,77,170,32,285,2.11,27.02,70.88,3.796,0.667
3,PE,PE4,It feels gratifying to help others by sharing ...,کمک به دیگران از طریق به‌اشتراک‌گذاری دانش در ...,1,6,57,180,41,285,2.46,20.00,77.54,3.891,0.670
4,PE,PE5,My interaction in knowledge sharing with other...,تعامل من با سایر دانشجویان در زمینه به‌اشتراک‌...,2,5,47,180,51,285,2.46,16.49,81.05,3.958,0.691
5,PE,PE6,I experience satisfaction when I share knowled...,هنگامی که دانش خود را در پلتفرم یادگیری الکترو...,0,4,59,175,47,285,1.40,20.70,77.89,3.930,0.652
6,PU,PU1,I find the e-learning platform highly motivati...,پلتفرم یادگیری الکترونیکی هنگام انجام وظایف طر...,2,13,66,170,34,285,5.26,23.16,71.58,3.775,0.740
7,PU,PU2,The e-learning platform proves to be efficient...,پلتفرم یادگیری الکترونیکی در مدیریت کارهای مرت...,2,5,66,166,46,285,2.46,23.16,74.39,3.874,0.716
8,PU,PU3,I experience ease when using the e-learning pl...,استفاده از پلتفرم یادگیری الکترونیکی برای انجا...,4,18,79,149,35,285,7.72,27.72,64.56,3.677,0.823
9,PEOU,PEOU1,Engaging with the e-learning platform feels st...,کار با پلتفرم یادگیری الکترونیکی برای من ساده ...,2,10,87,141,45,285,4.21,30.53,65.26,3.761,0.782


In [50]:
# در این سلول جدول پاسخ‌های سؤال‌به‌سؤال EFL در پوشه خود Dataset EFL و پوشه خروجی جداول ذخیره می‌شود.

os.makedirs(DATASET_EFL_FOLDER, exist_ok=True)
os.makedirs(TABLES_FOLDER, exist_ok=True)

response_table_EFL.to_excel(
    os.path.join(DATASET_EFL_FOLDER, "Dataset_EFL_Questionnaire_Response_Table.xlsx"),
    index=False
)

response_table_EFL.to_excel(
    os.path.join(TABLES_FOLDER, "Dataset_EFL_Questionnaire_Response_Table.xlsx"),
    index=False
)

print("EFL questionnaire response table saved successfully in both folders.")


EFL questionnaire response table saved successfully in both folders.


In [51]:
# در این سلول سازگاری معنایی سازه‌های مشترک Dataset B و EFL برای اعتبارسنجی بین‌دیتاستی مستند می‌شود.

harmonization_table = pd.DataFrame({

    "Construct": [
        "PU",
        "CON",
        "SAT",
        "CI"
    ],

    "Dataset_B": [
        "Perceived Usefulness",
        "Confirmation",
        "Satisfaction",
        "Continuance Intention"
    ],

    "Dataset_EFL": [
        "Perceived Usefulness",
        "Confirmation",
        "Satisfaction",
        "Continuance Intention"
    ],

    "Semantic_Compatibility": [
        "Low / Problematic",
        "Good",
        "Very High",
        "Very High"
    ],

    "Use_in_Core_Cross_Dataset_Model": [
        "No",
        "Yes",
        "Yes",
        "Yes - Target"
    ],

    "Reason": [
        "PU items measure substantially different content across the two questionnaires.",
        "Both measure confirmation of expectations, although individual item wording differs.",
        "The same core satisfaction concepts are measured, with mainly different item ordering.",
        "Both measure future regular, frequent, and continued use of e-learning."
    ]
})

harmonization_table

,Construct,Dataset_B,Dataset_EFL,Semantic_Compatibility,Use_in_Core_Cross_Dataset_Model,Reason
0,PU,Perceived Usefulness,Perceived Usefulness,Low / Problematic,No,PU items measure substantially different conte...
1,CON,Confirmation,Confirmation,Good,Yes,"Both measure confirmation of expectations, alt..."
2,SAT,Satisfaction,Satisfaction,Very High,Yes,The same core satisfaction concepts are measur...
3,CI,Continuance Intention,Continuance Intention,Very High,Yes - Target,"Both measure future regular, frequent, and con..."


In [52]:
# در این سلول جدول سازگاری مفهومی سازه‌های مشترک Dataset B و EFL ذخیره می‌شود.

os.makedirs(TABLES_FOLDER, exist_ok=True)

harmonization_table.to_excel(
    os.path.join(TABLES_FOLDER, "B_EFL_Construct_Harmonization.xlsx"),
    index=False
)

print("B-EFL harmonization table saved successfully.")


B-EFL harmonization table saved successfully.


In [53]:
# در این سلول مشخصات اصلی Dataset B و Dataset EFL پس از کنترل کیفیت در یک جدول مقایسه‌ای خلاصه می‌شوند.

dataset_comparison = pd.DataFrame({

    "Dataset": [
        "Dataset B",
        "Dataset EFL"
    ],

    "Role": [
        "Development / Training",
        "External Validation"
    ],

    "Original_N": [
        368,
        435
    ],

    "Final_N": [
        368,
        285
    ],

    "Exact_Duplicates_Removed": [
        0,
        105
    ],

    "Straight_Line_Removed": [
        0,
        45
    ],

    "Missing_Values": [
        0,
        0
    ],

    "Likert_Scale": [
        "1-5",
        "1-5"
    ],

    "Core_Predictors": [
        "CON, SAT",
        "CON, SAT"
    ],

    "Target": [
        "CI",
        "CI"
    ],

    "CON_Alpha": [
        0.905,
        0.821
    ],

    "SAT_Alpha": [
        0.911,
        0.780
    ],

    "CI_Alpha": [
        0.915,
        0.836
    ]
})

dataset_comparison

,Dataset,Role,Original_N,Final_N,Exact_Duplicates_Removed,Straight_Line_Removed,Missing_Values,Likert_Scale,Core_Predictors,Target,CON_Alpha,SAT_Alpha,CI_Alpha
0,Dataset B,Development / Training,368,368,0,0,0,1-5,"CON, SAT",CI,0.905,0.911,0.915
1,Dataset EFL,External Validation,435,285,105,45,0,1-5,"CON, SAT",CI,0.821,0.780,0.836


In [54]:
# در این سلول بررسی می‌شود آیا در Dataset B پاسخ‌دهنده‌ای به تمام آیتم‌های پرسشنامه یک امتیاز یکسان داده است یا خیر.

straightline_mask_B = (
    df_B[questionnaire_columns_B]
    .nunique(axis=1)
    .eq(1)
)

straightline_count_B = straightline_mask_B.sum()

print("Total rows in Dataset B:", len(df_B))
print("Straight-line responses:", straightline_count_B)
print(
    "Percentage of straight-line responses:",
    round(straightline_count_B / len(df_B) * 100, 2),
    "%"
)

df_B.loc[
    straightline_mask_B,
    questionnaire_columns_B
].head(20)

Total rows in Dataset B: 368
Straight-line responses: 79
Percentage of straight-line responses: 21.47 %


,PEdS1,PEdS2,PEdS3,PEdS4,PEmS1,PEmS2,PEmS3,PEmS4,PU1,PU2,PU3,CON1,CON2,CON3,SAT1,SAT2,SAT3,CI1,CI2,CI3
5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
8,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
15,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
19,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
22,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
31,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
32,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
33,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
41,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
42,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3


In [55]:
# در این سلول Duplicateهای Dataset B بدون ستون شناسه و زمان تکمیل پرسشنامه بررسی می‌شوند.

duplicate_check_columns_B = [
    "Gender",
    "Age",
    "Grade",
    "Subject"
] + questionnaire_columns_B

duplicate_rows_B_without_metadata = (
    df_B[duplicate_check_columns_B]
    .duplicated()
    .sum()
)

unique_rows_B_without_metadata = (
    df_B[duplicate_check_columns_B]
    .drop_duplicates()
    .shape[0]
)

print("Total rows:", len(df_B))
print(
    "Unique rows excluding index and totalseconds:",
    unique_rows_B_without_metadata
)
print(
    "Duplicated rows excluding index and totalseconds:",
    duplicate_rows_B_without_metadata
)

Total rows: 368
Unique rows excluding index and totalseconds: 303
Duplicated rows excluding index and totalseconds: 65


In [56]:
# در این سلول بررسی می‌شود الگوهای تکراری Dataset B تا چه اندازه با پاسخ‌های Straight-line هم‌پوشانی دارند.

duplicate_group_mask_B = (
    df_B[duplicate_check_columns_B]
    .duplicated(keep=False)
)

duplicate_extra_mask_B = (
    df_B[duplicate_check_columns_B]
    .duplicated(keep="first")
)

duplicate_group_rows_B = duplicate_group_mask_B.sum()
duplicate_extra_rows_B = duplicate_extra_mask_B.sum()

overlap_duplicate_straightline_B = (
    duplicate_group_mask_B & straightline_mask_B
).sum()

non_straightline_duplicate_group_B = (
    duplicate_group_mask_B & ~straightline_mask_B
).sum()

print("Total rows:", len(df_B))
print("Rows belonging to a repeated complete profile:", duplicate_group_rows_B)
print("Extra duplicated rows:", duplicate_extra_rows_B)
print("Straight-line rows:", straightline_count_B)

print(
    "Straight-line rows inside repeated profiles:",
    overlap_duplicate_straightline_B
)

print(
    "Non-straight-line rows inside repeated profiles:",
    non_straightline_duplicate_group_B
)

Total rows: 368
Rows belonging to a repeated complete profile: 82
Extra duplicated rows: 65
Straight-line rows: 79
Straight-line rows inside repeated profiles: 70
Non-straight-line rows inside repeated profiles: 12


In [57]:
# در این سلول زمان تکمیل پرسشنامه در پاسخ‌های Straight-line با سایر پاسخ‌دهندگان Dataset B مقایسه می‌شود.

time_comparison_B = pd.DataFrame({
    "All_Respondents": df_B["totalseconds"].describe(),
    "Straight_Line": df_B.loc[
        straightline_mask_B, "totalseconds"
    ].describe(),
    "Non_Straight_Line": df_B.loc[
        ~straightline_mask_B, "totalseconds"
    ].describe()
})

time_comparison_B

,All_Respondents,Straight_Line,Non_Straight_Line
count,368.000000,79.000000,289.000000
mean,139.081522,83.822785,154.186851
std,269.083334,39.989024,301.276462
min,27.000000,27.000000,32.000000
25%,71.000000,55.000000,75.000000
50%,100.000000,76.000000,106.000000
75%,137.500000,107.500000,148.000000
max,3779.000000,244.000000,3779.000000


In [58]:
# در این سلول پاسخ‌های بسیار سریع و Straight-line در Dataset B ترکیب می‌شوند تا پاسخ‌های پرریسک شناسایی شوند.

median_time_B = df_B["totalseconds"].median()

df_B_quality = df_B.copy()

# نسبت سرعت هر فرد نسبت به زمان میانه کل نمونه
df_B_quality["Speed_Ratio"] = (
    median_time_B / df_B_quality["totalseconds"]
)

# پاسخ بسیار سریع: بیش از دو برابر سریع‌تر از پاسخ‌دهنده معمولی
df_B_quality["Very_Fast"] = (
    df_B_quality["Speed_Ratio"] > 2
)

# Straight-line
df_B_quality["Straight_Line"] = straightline_mask_B.values

# پاسخ پرریسک: هم Straight-line و هم بسیار سریع
df_B_quality["High_Risk_Response"] = (
    df_B_quality["Very_Fast"]
    & df_B_quality["Straight_Line"]
)

print("Median completion time:", median_time_B, "seconds")
print("Approximate very-fast cutoff:", median_time_B / 2, "seconds")

print("\nVery-fast responses:", df_B_quality["Very_Fast"].sum())
print("Straight-line responses:", df_B_quality["Straight_Line"].sum())
print(
    "Straight-line AND very-fast responses:",
    df_B_quality["High_Risk_Response"].sum()
)

Median completion time: 100.0 seconds
Approximate very-fast cutoff: 50.0 seconds

Very-fast responses: 22
Straight-line responses: 79
Straight-line AND very-fast responses: 14


In [59]:
# در این سلول پاسخ‌های Straight-line کامل از یک نسخه جداگانه Dataset B حذف می‌شوند و داده خام بدون تغییر باقی می‌ماند.

df_B_clean = df_B.loc[
    ~straightline_mask_B
].copy()

print("Original Dataset B rows:", len(df_B))
print("Straight-line responses removed:", straightline_count_B)
print("Final cleaned Dataset B rows:", len(df_B_clean))

Original Dataset B rows: 368
Straight-line responses removed: 79
Final cleaned Dataset B rows: 289


In [60]:
# در این سلول کیفیت Dataset B پس از حذف Straight-lineها دوباره بررسی می‌شود.

remaining_straightlines_B = (
    df_B_clean[questionnaire_columns_B]
    .nunique(axis=1)
    .eq(1)
    .sum()
)

remaining_repeated_profiles_B = (
    df_B_clean[duplicate_check_columns_B]
    .duplicated()
    .sum()
)

print("Final cleaned rows:", len(df_B_clean))
print("Remaining straight-line responses:", remaining_straightlines_B)
print("Remaining repeated complete profiles:", remaining_repeated_profiles_B)

Final cleaned rows: 289
Remaining straight-line responses: 0
Remaining repeated complete profiles: 6


In [61]:
# در این سلول الگوهای تکراری باقی‌مانده در Dataset B بررسی می‌شوند تا مشخص شود آیا رکوردهای مستقلی با index و زمان پاسخ متفاوت هستند.

remaining_duplicate_mask_B = (
    df_B_clean[duplicate_check_columns_B]
    .duplicated(keep=False)
)

remaining_duplicate_rows_B = df_B_clean.loc[
    remaining_duplicate_mask_B,
    ["index", "totalseconds"] + duplicate_check_columns_B
].sort_values(
    by=duplicate_check_columns_B
)

print(
    "Number of rows belonging to remaining repeated profiles:",
    len(remaining_duplicate_rows_B)
)

display(remaining_duplicate_rows_B)

Number of rows belonging to remaining repeated profiles: 12


,index,totalseconds,Gender,Age,Grade,Subject,PEdS1,PEdS2,PEdS3,PEdS4,...,PU3,CON1,CON2,CON3,SAT1,SAT2,SAT3,CI1,CI2,CI3
348,349,107,1,2,1,2,4,5,5,3,...,4,2,3,3,3,4,4,4,3,1
349,350,146,1,2,1,2,4,5,5,3,...,4,2,3,3,3,4,4,4,3,1
63,64,52,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,3,3,3
134,135,127,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,3,3,3
86,87,68,2,2,1,1,4,4,5,5,...,5,5,5,5,5,5,5,5,5,5
252,253,64,2,2,1,1,4,4,5,5,...,5,5,5,5,5,5,5,5,5,5
121,122,90,2,2,1,1,4,5,5,5,...,5,5,5,5,5,5,5,5,5,5
331,332,56,2,2,1,1,4,5,5,5,...,5,5,5,5,5,5,5,5,5,5
275,276,153,2,2,1,2,4,3,3,5,...,4,3,3,3,3,3,3,4,4,5
276,277,153,2,2,1,2,4,3,3,5,...,4,3,3,3,3,3,3,4,4,5


In [62]:
# در این سلول Duplicateهای واقعی Dataset B با نادیده گرفتن فقط ستون index بررسی می‌شوند؛
# زمان تکمیل پرسشنامه و تمام متغیرهای دیگر در مقایسه باقی می‌مانند.

duplicate_check_strict_B = [
    column for column in df_B.columns
    if column != "index"
]

strict_duplicate_mask_B = (
    df_B[duplicate_check_strict_B]
    .duplicated(keep="first")
)

strict_duplicate_group_mask_B = (
    df_B[duplicate_check_strict_B]
    .duplicated(keep=False)
)

print(
    "Strict duplicate rows excluding only index:",
    strict_duplicate_mask_B.sum()
)

display(
    df_B.loc[
        strict_duplicate_group_mask_B,
        ["index"] + duplicate_check_strict_B
    ]
)

Strict duplicate rows excluding only index: 6


,index,totalseconds,Gender,Age,Grade,Subject,PEdS1,PEdS2,PEdS3,PEdS4,...,PU3,CON1,CON2,CON3,SAT1,SAT2,SAT3,CI1,CI2,CI3
58,59,66,2,2,1,1,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
60,61,66,2,2,1,1,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
80,81,46,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
84,85,53,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
88,89,59,1,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
90,91,53,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
128,129,65,2,2,1,1,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
160,161,46,2,2,1,1,4,4,4,4,...,4,4,4,4,4,4,4,4,4,4
275,276,153,2,2,1,2,4,3,3,5,...,4,3,3,3,3,3,3,4,4,5
276,277,153,2,2,1,2,4,3,3,5,...,4,3,3,3,3,3,3,4,4,5


In [63]:
# در این سلول برای Dataset B و EFL پرچم‌های کنترل کیفیت ساخته می‌شوند، بدون اینکه هیچ رکوردی از داده خام حذف شود.

# -------------------------
# Dataset B
# -------------------------

df_B_flagged = df_B.copy()

df_B_flagged["Straight_Line_Flag"] = (
    df_B_flagged[questionnaire_columns_B]
    .nunique(axis=1)
    .eq(1)
)

median_time_B = df_B_flagged["totalseconds"].median()

df_B_flagged["Very_Fast_Flag"] = (
    df_B_flagged["totalseconds"] < (median_time_B / 2)
)

df_B_flagged["High_Risk_Flag"] = (
    df_B_flagged["Straight_Line_Flag"]
    & df_B_flagged["Very_Fast_Flag"]
)

strict_columns_B = [
    column for column in df_B.columns
    if column != "index"
]

df_B_flagged["Strict_Duplicate_Flag"] = (
    df_B[strict_columns_B]
    .duplicated(keep=False)
)


# -------------------------
# Dataset EFL
# -------------------------

df_EFL_flagged = df_EFL.copy()

df_EFL_flagged["Straight_Line_Flag"] = (
    df_EFL_flagged[questionnaire_columns_EFL]
    .nunique(axis=1)
    .eq(1)
)

df_EFL_flagged["Exact_Duplicate_Flag"] = (
    df_EFL.duplicated(keep=False)
)


print("Dataset B flagged shape:", df_B_flagged.shape)
print("Dataset EFL flagged shape:", df_EFL_flagged.shape)

print("\nDataset B quality flags:")
print(df_B_flagged[
    [
        "Straight_Line_Flag",
        "Very_Fast_Flag",
        "High_Risk_Flag",
        "Strict_Duplicate_Flag"
    ]
].sum())

print("\nDataset EFL quality flags:")
print(df_EFL_flagged[
    [
        "Straight_Line_Flag",
        "Exact_Duplicate_Flag"
    ]
].sum())

Dataset B flagged shape: (368, 30)
Dataset EFL flagged shape: (435, 34)

Dataset B quality flags:
Straight_Line_Flag       79
Very_Fast_Flag           22
High_Risk_Flag           14
Strict_Duplicate_Flag    12
dtype: int64

Dataset EFL quality flags:
Straight_Line_Flag      108
Exact_Duplicate_Flag    146
dtype: int64


In [64]:
# در این سلول خلاصه نهایی کنترل کیفیت ساخته و نسخه‌های Quality-Flagged برای Notebookهای بعدی ذخیره می‌شوند.

quality_audit_summary = pd.DataFrame({
    "Dataset": ["Dataset B", "Dataset EFL", "Dataset C"],
    "Original_N": [len(df_B), len(df_EFL), len(df_C)],
    "Missing_Values": [
        int(df_B.isnull().sum().sum()),
        int(df_EFL.isnull().sum().sum()),
        int(df_C.isnull().sum().sum())
    ],
    "Straight_Line_Responses": [
        int(df_B_flagged["Straight_Line_Flag"].sum()),
        int(df_EFL_flagged["Straight_Line_Flag"].sum()),
        "Not used for final decision"
    ],
    "Duplicate_Issue": [
        int(df_B[strict_columns_B].duplicated().sum()),
        int(df_EFL.duplicated().sum()),
        int(df_C.duplicated().sum())
    ],
    "Current_Status": [
        "Development dataset - retained with quality flags",
        "External validation candidate - retained with quality flags",
        "Excluded because of extensive complete duplication"
    ]
})

display(quality_audit_summary)

os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(TABLES_FOLDER, exist_ok=True)

df_B_flagged.to_csv(
    os.path.join(PROCESSED_FOLDER, "dataset_B_quality_flagged.csv"),
    index=False
)

df_EFL_flagged.to_csv(
    os.path.join(PROCESSED_FOLDER, "dataset_EFL_quality_flagged.csv"),
    index=False
)

quality_audit_summary.to_excel(
    os.path.join(TABLES_FOLDER, "Data_Quality_Audit_Summary.xlsx"),
    index=False
)

print("\nQuality-control outputs saved successfully.")


,Dataset,Original_N,Missing_Values,Straight_Line_Responses,Duplicate_Issue,Current_Status
0,Dataset B,368,0,79,6,Development dataset - retained with quality flags
1,Dataset EFL,435,0,108,105,External validation candidate - retained with ...
2,Dataset C,166,0,Not used for final decision,123,Excluded because of extensive complete duplica...



Quality-control outputs saved successfully.
